# 4.1. Evaluation metrics: session overview

## What we cover in this module 
**Accuracy**, and why a dummy model that predicts "no churn" for everybody is a useful sanity check \
\
The **<u>`confusion table`</u>**: the four types of correct decisions and errors a binary classifier makes \
\
**<u>`Precision and recall`</u>**: metrics that look at the positive class specifically \
\
**<u>`ROC curves`</u>**: true positive rate and false positive rate across all possible thresholds, with random and ideal models as reference points \
\
**<u>`ROC AUC`</u>**: the area under the ROC curve, a single number for curve quality \
\
**<u>`Cross-validation`</u>**: a more reliable way to measure model quality and tune parameters \
The whole module walks through one notebook, notebook.ipynb - the same churn prediction project as in module 3. 

### Notes
The fourth week of Machine Learning Zoomcamp is about different metrics to evaluate a binary classifier. These measures include accuracy, confusion table, precision, recall, ROC curves(TPR, FRP, random model, and ideal model), AUROC, and cross-validation. \
 \
For this project, we used a Kaggle dataset about churn prediction.

In [54]:
# LESSON 3 RECAP
# Import packages
import numpy                      as     np
import pandas                     as     pd
import matplotlib.pyplot          as     plt
from   sklearn.linear_model       import LogisticRegression
from   sklearn.model_selection    import train_test_split
from   sklearn.feature_extraction import DictVectorizer
from   sklearn.metrics            import accuracy_score
from   collections                import Counter

#   First, we load the data and clean it: 
#   - column names are lowercased, 
#   - spaces are replaced with underscores, 
#   - totalcharges is converted to numbers with errors='coerce' and 
#   - the missing values are filled with 0, and 
#   - the target is turned into 0/1:

# LOAD PACKAGES TO DOWNLOAD KAGGLE DATA
import glob       # Finds files matching a pattern
import kagglehub  # Downloads datasets from Kaggle

# DOWNLOAD THE DATASET; 'KaggleHub' RETURNS A FOLDER CONTAINING FILES
dataset_path    = kagglehub.dataset_download("blastchar/telco-customer-churn")

# FIND THE CSV INSIDE OF THE DOWNLOADED FOLDER (INCLUDING SUBFOLDERS)
csv_path        = glob.glob(
    f"{dataset_path}/**/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    recursive=True
)[0]  # INDEX [0] SAYS TAKE THE FIRST FILE TO MATCH THE PATTERN

# LOAD THE CSV INTO A 'pandas' DATAFRAME
df              = pd.read_csv(csv_path)

# CONVERT COLUMN NAMES TO LOWERCASE AND REPLACE WHITESPACES WITH UNDERSCORES
df.columns      = df.columns.str.lower().str.replace(' ', '_')

# CREATE A LIST OF CATEGORICAL DATA COLUMNS
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)

# CREATE A LIST OF NUMERICAL DATA COLUMNS
numerical_columns = list(df.dtypes[df.dtypes.isin([np.dtype("int64"), np.dtype("float64")])].index)

# REPLACE WHITESPACES IN DATA WITH UNDERSCORES FOR CATEGORICAL DATA COLUMNS
for c in categorical_columns:
    df[c]       = df[c].str.lower().str.replace(' ', '_')

# FILL EMPTY CELLS WITH 'NaN'
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')

# CONVERT 'NaN' VALUES TO ZEROS
df.totalcharges = df.totalcharges.fillna(0)

# REPLACE 'yes' VALUES IN THE 'churn' COLUMN WITH ONES
#   ALL OTHER VALUES WILL BECOME ZEROS
df.churn        = (df.churn == 'yes').astype(int)


# Then we split the data into train, validation and test: 
#   - 60% for training, 20% for validation and 20% for test. 
# We do it in two steps 
#   - first split off the test set, then 
#   - split the rest - and 
# we use random_state=1 so the shuffle is reproducible:

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val       = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [55]:
# DEBUG
# df.dtypes

In [56]:
# DEBUG
# categorical_columns

In [57]:
# DEBUG
# numerical_columns

In [58]:
# CREATE THE TRAINING DATASET

# IMPORT FUNCTION USED TO APPY ONE-HOT ENCODING (OHE)
from sklearn.feature_extraction import DictVectorizer

# CREATE 'dv' MATRIX 
dv         = DictVectorizer(sparse=False) 

# CREATE TRAINING DICTIONARY
train_dict = df_train[categorical_columns + numerical_columns].to_dict(orient='records')

# CREATE FEATURE MATRIX WITH ONE-HOT ENCODED COLUMNS
X_train    = dv.fit_transform(train_dict)

# CREATE TARGET VECTOR
y_train    = df_train.churn.values

In [59]:
# CREATE THE MODEL
model = LogisticRegression(solver='lbfgs' , max_iter=10000)
# DEFAULT SOLVER IN NEWER VERSIONS OF 'sklearn': solver='lbfgs'
#   MUST SPECIFY SOLVER EXPLICITLY IN OLDER VERSIONS

# 'fit()' METHOD TRAINS THE MODEL
model.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",10000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use 

In [60]:
# CREATE VALIDATION DATASET
# 1. ENCODE THE VALIDATION DATASET
val_dict = df_val[categorical_columns + numerical_columns].to_dict(orient='records')
X_val    = dv.fit_transform(val_dict)
y_val    = df_val.churn.values

In [62]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape

((4225, 4271), (4225,), (1409, 1455), (1409,))

In [53]:
# We keep 3 numerical features (tenure, monthlycharges, totalcharges) and 16 categorical ones, 
#   one-hot encode them with 
#     DictVectorizer(sparse=False), and 
#   train a logistic regression model.
#
# At the end of the recap we get predictions on the validation set. 
#     predict_proba returns the churn probability for each customer, and 
#    we say the model predicts churn when this probability is at least 0.5:

y_val_pred     = model.predict_proba(X_val)[:, 1]
churn_decision = (y_val_pred >= 0.5)
(y_val_pred == churn_decision).mean()

ValueError: X has 1455 features, but LogisticRegression is expecting 4271 features as input.

In [52]:
# DEBUG
y_pred

array([6.72293171e-04, 3.69581547e-03, 3.80735523e-03, ...,
       3.07743235e-03, 9.96459215e-01, 9.96383871e-01], shape=(1409,))

## 4.2. Accuracy
Accuracy measures the fraction of correct predictions: the number of correct predictions divided by the total number of predictions.

In [ ]:
# CHECK DATASET SIZE
len(y_val)

# SHOULD BE 1,409

In [ ]:
# ASSESS WHETHER PREDICTION WAS CORRECT
(y_val == churn_decision).mean()

# SHOULD BE 1,132

In [ ]:
# # SCI-KIT HAS A BUILT-IN ACCURACY SCORE 
from sklearn.metrics import accuracy_score

PRINT(f"manual accuracy score calculation:  {(y_val == churn_decision).mean()}")
PRINT(f"sklearn accuracy score calculation: {accuracy_score(y_val, y_pred >= 0.5)}")
# accuracy_score(y_val, y_pred >= 0.5)

#SHOULD RETURN 0.8034066713981547

## Checking different thresholds
The 0.5 threshold was a guess. Nothing forces us to use it: the model outputs a probability, and we can announce churn for any cutoff we like. Let's check all thresholds from 0 to 1 in steps of 0.05 and see which one gives the best accuracy:

In [ ]:
thresholds = np.linspace(0, 1, 21)

scores = []

for t in thresholds:
    score = accuracy_score(y_val, y_pred >= t)
    print('%.2f %.3f' % (t, score))
    scores.append(score)

# SHOULD RETURN
#  THRESHHOLD SCORE
#   0.00       0.274
#   0.05       0.509
#   0.10       0.591
#   0.15       0.666
#   0.20       0.710
#   0.25       0.739
#   0.30       0.760
#   0.35       0.772
#   0.40       0.785
#   0.45       0.793
#   0.50       0.803
#   0.55       0.801
#   0.60       0.795
#   0.65       0.786
#   0.70       0.766
#   0.75       0.744
#   0.80       0.735
#   0.85       0.726
#   0.90       0.726
#   0.95       0.726
#   1.00       0.726

So for this problem the best decision cutoff is indeed 0.5, with 80% accuracy. That was a lucky guess - but it didn't have to be. On other problems the best threshold is often different from 0.5, and this simple loop is how you find it.

In [ ]:
# 
from collections import Counter

Counter(y_pred >= 1.0)

In [ ]:
# DETERMINE COUNT OF iNVALID PREDICTIONS (PREDICTION > 100%)
#   EVERY PREDICTION SHOULD BE FALSE AS NOTHING CAN BE PREDICTED WITH GREATER THAN 100% CORRECTNESS
Counter({False: 1409})

In [ ]:
# What accuracy does this dummy model get? The answer is the fraction of customers who did not churn:

1 - y_val.mean()

# ANSWER SHOULD BE 0.7260468417317246 - 73%.

A model that doesn't look at the data at all scores 73%, and our logistic regression scores 80%.  \
The improvement over the dummy baseline is much smaller than "80% correct" made it sound.

## Why accuracy fails here: class imbalance

The reason is that the classes are unbalanced: 
* only 27% of the customers in the validation set churned, and
* 73% stayed.

\
This is called class imbalance - there are many more instances of one class than of the other.

With such an imbalance, a model can get a high accuracy by always predicting the majority class, while being completely useless for the minority class - which is exactly the class we care about: we want to find the customers who are about to leave.

\
So for this problem, accuracy alone cannot tell us how good the model is. We need metrics that look at the errors in more detail - that is what the confusion table in the next lesson is for.

## Notes

__<u>Accuracy measures the fraction of correct predictions</u>.__ Specifically, it is the number of correct predictions divided by the total number of predictions. \
\
We can change the decision threshold, it should not be always 0.5. But, in this particular problem, the best decision cutoff, associated with the hightest accuracy (80%), was indeed 0.5. \
\
Note that if we build a dummy model in which the decision cutoff is 1, so the algorithm predicts that no clients will churn, the accuracy would be 73%. Thus, we can see that the improvement of the original model with respect to the dummy model is not as high as we would expect. \
\
Therefore, in this problem accuracy can not tell us how good is the model because the dataset is unbalanced, which means that there are more instances from one category than the other. This is also known as class imbalance.

### Classes and methods:

__<u>np.linspace(x,y,z)</u>__ - returns a numpy array starting at x until y with z evenly spaced samples \
\
__<u>Counter(x)</u>__ - collection class that counts the number of instances that satisfy the x condition \
\
__<u>accuracy_score(x, y)</u>__ - sklearn.metrics class for calculating the accuracy of a model, given a predicted x dataset and a target y dataset.


## 4.3. Confusion table
Accuracy collapses all predictions into a single number, so it cannot tell us which kinds of mistakes the model makes. \
 \
In this lesson we look at the **confusion table**: a way of measuring the different types of errors and correct decisions a binary classifier can make, arranged in one table.

### Four outcomes of a prediction

For churn prediction, each customer in the validation set falls into one of four categories, depending on what we predicted and what actually happened: \
 
| Prediction 	    |  Actual 	     | Name                 | 
| ------------------|----------------|----------------------| 
| Customer will churn (positive) 	|  Churned 	     | True Positive (TP)   | 
| Customer will churn (positive)	|  Did not churn | False Positive (FP)  | 
| Customer will not churn (negative) |  Churned 	     |  False Negative (FN) | 
| Customer will not churn (negative) |  Did not churn |  True Negative (TN)  | 

 
### Computing the counts ###

To put every customer into one of the four cells, we compare the actual values with the predictions at threshold 0.5:


In [ ]:

actual_positive = (y_val == 1)
actual_negative = (y_val == 0)

t = 0.5
predict_positive = (y_pred >= t)
predict_negative = (y_pred < t)